# KioGuard FP16 gate 재평가 v3.1

이 노트북은 1차 20문장 gate의 오류 분석 후 만든 prompt v3.1로 상위 후보를 재평가합니다. 기존 FP16_COMMON_V3 결과는 수정하지 않으며 새 디렉터리에 원출력과 진단을 저장합니다.


In [ ]:
# 셀 1: 실행 설정. Drive mount 문제가 있으면 기본 로컬 저장을 사용합니다.
USE_GOOGLE_DRIVE = False
MODEL_KEY = "qwen35_2b"  # qwen35_2b | qwen2_15b | gemma2_2b
RUN_SCOPE = "gate"       # 이 노트북에서는 먼저 gate만 실행
PROMPT_VERSION = "v3_1"
MAX_NEW_TOKENS = 220
SEED = 20260930

REPO_URL = "https://github.com/cjuproject34/Kioguard.git"
REPO_DIR = "/content/KioGuard"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/KioGuard/experiments/FP16_GATE_V3_1"
else:
    OUTPUT_ROOT = "/content/KioGuard/experiments/FP16_GATE_V3_1"

print({"MODEL_KEY": MODEL_KEY, "RUN_SCOPE": RUN_SCOPE, "PROMPT_VERSION": PROMPT_VERSION, "OUTPUT_ROOT": OUTPUT_ROOT})


In [ ]:
# 셀 2: 저장소와 평가 자료 확보
import os, subprocess

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

EVAL_ROOT = os.path.join(REPO_DIR, "research", "llm-intent", "evaluation")
assert os.path.exists(os.path.join(EVAL_ROOT, "dataset_manifest.json")), "평가 자료가 저장소에 없습니다."
print("EVAL_ROOT:", EVAL_ROOT)


In [ ]:
# 셀 3: 런타임 검증. 설치 셀이 아니라 현재 환경을 확인하는 셀입니다.
import importlib.metadata as mdlib
import torch, torchvision, torchaudio, transformers
from transformers import AutoModelForCausalLM, AutoModelForMultimodalLM, AutoProcessor, AutoTokenizer

versions = {
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "torchvision": torchvision.__version__,
    "torchaudio": torchaudio.__version__,
    "transformers": transformers.__version__,
    "transformers_commit": getattr(transformers, "__commit__", None),
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(versions)
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다."
assert "+cu128" in torch.__version__, "현재 검증 기준은 torch cu128입니다. 환경을 다시 확인하세요."
assert "+cu128" in torchvision.__version__, "torchvision CUDA 버전 불일치"
assert "+cu128" in torchaudio.__version__, "torchaudio CUDA 버전 불일치"
print("모든 import 성공")


In [ ]:
# 셀 4: 모델 행렬과 평가 데이터 로드
import json, pathlib

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

with open(os.path.join(EVAL_ROOT, "configs", "model_matrix.json"), encoding="utf-8") as f:
    matrix = json.load(f)
model_cfg = next(m for m in matrix["models"] if m["key"] == MODEL_KEY)

core_path = os.path.join(EVAL_ROOT, "data", "selection_dev_v0.1.jsonl")
challenge_path = os.path.join(EVAL_ROOT, "data", "selection_challenge_v0.1.jsonl")
legacy_path = os.path.join(EVAL_ROOT, "data", "legacy48_regression_v1.2.jsonl")
core = read_jsonl(core_path)
challenge = read_jsonl(challenge_path)
legacy = read_jsonl(legacy_path)

gate_ids = set()
for intent in matrix.get("intent_order", []):
    pass
for intent in ["SELECT", "MODIFY", "FULFILLMENT", "PAYMENT_METHOD", "RECEIPT", "BENEFIT_CONTROL", "NAVIGATE", "CANCEL", "GUIDE", "INFO", "UI_CONTROL", "HELP", "COMMIT_REQUEST", "RESET"]:
    gate_ids.add(next(x["case_id"] for x in core if any(r["intent"] == intent for r in x["gold"]["requests"])))
gate_ids.update(["CH-CLR-01", "CH-CLR-02", "CH-NEG-01", "CH-NEG-02", "CH-MUL-01", "CH-CTX-04"])

if RUN_SCOPE == "gate":
    cases = [x for x in core + challenge if x["case_id"] in gate_ids]
elif RUN_SCOPE == "selection":
    cases = core + challenge
elif RUN_SCOPE == "all":
    cases = core + challenge + legacy
else:
    raise ValueError("RUN_SCOPE must be gate, selection, or all")

print("model:", model_cfg)
print("cases:", len(cases), "core/challenge/legacy:", len(core), len(challenge), len(legacy))


In [ ]:
# 셀 5: prompt v3.1 파일 고정
import hashlib

prompt_path = os.path.join(EVAL_ROOT, "prompts", "prompt_v3_1_full.txt")
with open(prompt_path, encoding="utf-8") as f:
    PROMPT_PREFIX = f.read().strip() + "\n"
PROMPT_HASH = hashlib.sha256(PROMPT_PREFIX.encode("utf-8")).hexdigest()

def build_prompt(case):
    payload = {"context": case["context"], "utterance": case["utterance"]}
    return PROMPT_PREFIX + "\n입력:\n" + json.dumps(payload, ensure_ascii=False, separators=(",", ":"))

print("prompt_version:", PROMPT_VERSION)
print("prompt_sha256:", PROMPT_HASH)
print(build_prompt(cases[0])[:1800])


In [ ]:
# 셀 6: Hugging Face 인증 토큰을 안전하게 가져옵니다. 토큰은 출력하지 않습니다.
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if model_cfg["model_id"].startswith("google/") and not hf_token:
    print("Gemma gated 모델이면 Colab Secrets에 HF_TOKEN을 등록해야 할 수 있습니다.")
else:
    print("인증 준비 완료")


In [ ]:
# 셀 7: 선택 모델 하나 로드
import gc, os, psutil, time

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
process = psutil.Process(os.getpid())
rss_before = process.memory_info().rss / 1024**3
load_start = time.perf_counter()

common_kwargs = {
    "revision": model_cfg["revision"],
    "torch_dtype": torch.float16,
    "device_map": "auto",
    "low_cpu_mem_usage": True,
}
if hf_token:
    common_kwargs["token"] = hf_token

if model_cfg["loader"] == "multimodal":
    processor = AutoProcessor.from_pretrained(model_cfg["model_id"], revision=model_cfg["revision"], token=hf_token)
    model = AutoModelForMultimodalLM.from_pretrained(model_cfg["model_id"], **common_kwargs)
    tokenizer = None
else:
    tokenizer = AutoTokenizer.from_pretrained(model_cfg["model_id"], revision=model_cfg["revision"], token=hf_token)
    processor = None
    model = AutoModelForCausalLM.from_pretrained(model_cfg["model_id"], **common_kwargs)

model.eval()
torch.cuda.synchronize()
load_seconds = time.perf_counter() - load_start
rss_after = process.memory_info().rss / 1024**3
resolved_revision = getattr(model.config, "_commit_hash", None)

load_metrics = {
    "model_class": model.__class__.__name__,
    "resolved_revision": resolved_revision,
    "load_seconds": load_seconds,
    "process_rss_before_gib": rss_before,
    "process_rss_after_gib": rss_after,
    "gpu_allocated_gib": torch.cuda.memory_allocated() / 1024**3,
    "gpu_reserved_gib": torch.cuda.memory_reserved() / 1024**3,
    "gpu_peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "parameter_count": sum(p.numel() for p in model.parameters()),
}
print(json.dumps(load_metrics, ensure_ascii=False, indent=2))


In [ ]:
# Cell 8: native chat template + greedy generation
def render_chat(prompt):
    if processor is not None:
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        try:
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            thinking_control = "enable_thinking_false"
        except TypeError:
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            thinking_control = "unsupported"
        batch = processor(text=[text], return_tensors="pt", add_special_tokens=False)
        decode = processor.decode
    else:
        messages = [{"role": "user", "content": prompt}]
        try:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            thinking_control = "enable_thinking_false"
        except TypeError:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            thinking_control = "unsupported"
        batch = tokenizer(text, return_tensors="pt", add_special_tokens=False)
        decode = tokenizer.decode
    device = next(model.parameters()).device
    batch = {k: v.to(device) if hasattr(v, "to") else v for k, v in batch.items()}
    return batch, decode, thinking_control

def first_token_id(value):
    if isinstance(value, (list, tuple)):
        return value[0] if value else None
    return value

@torch.inference_mode()
def generate_one(case):
    prompt = build_prompt(case)
    batch, decode, thinking_control = render_chat(prompt)
    input_tokens = int(batch["input_ids"].shape[-1])
    pad_token_id = first_token_id(
        getattr(getattr(model, "generation_config", None), "pad_token_id", None)
    )
    if pad_token_id is None:
        pad_token_id = first_token_id(
            getattr(getattr(model, "generation_config", None), "eos_token_id", None)
        )
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    output = model.generate(
        **batch,
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=pad_token_id,
    )
    torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    generated = output[0, input_tokens:]
    raw = decode(generated, skip_special_tokens=True).strip()
    return {
        "case_id": case["case_id"],
        "raw_output": raw,
        "input_tokens": input_tokens,
        "output_tokens": int(generated.numel()),
        "generation_seconds": seconds,
        "gpu_peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
        "thinking_control": thinking_control,
    }

preview = generate_one(cases[0])
print(json.dumps(preview, ensure_ascii=False, indent=2))


In [ ]:
# 셀 9: 전체 선택 범위 실행. 사례마다 즉시 Drive에 저장하므로 중단돼도 앞 결과는 남습니다.
import datetime, statistics

run_name = f"{MODEL_KEY}_{RUN_SCOPE}_fp16_v3_1"
run_dir = os.path.join(OUTPUT_ROOT, run_name)
os.makedirs(run_dir, exist_ok=True)
pred_path = os.path.join(run_dir, "predictions.jsonl")

completed = set()
if os.path.exists(pred_path):
    with open(pred_path, encoding="utf-8") as f:
        completed = {json.loads(line)["case_id"] for line in f if line.strip()}
print("resume completed:", len(completed))

results = []
with open(pred_path, "a", encoding="utf-8") as f:
    for idx, case in enumerate(cases, 1):
        if case["case_id"] in completed:
            continue
        try:
            row = generate_one(case)
            row["status"] = "SUCCESS"
        except Exception as exc:
            row = {"case_id": case["case_id"], "raw_output": "", "status": "ERROR", "error_type": type(exc).__name__, "error": str(exc)}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        results.append(row)
        print(f"[{idx}/{len(cases)}] {case['case_id']} {row['status']} {row.get('generation_seconds', 0):.3f}s")

with open(pred_path, encoding="utf-8") as f:
    saved_rows = [json.loads(line) for line in f if line.strip()]

manifest = {
    "experiment_id": "FP16_GATE_V3_1",
    "prompt_version": PROMPT_VERSION,
    "run_name": run_name,
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model_key": MODEL_KEY,
    "model_id": model_cfg["model_id"],
    "requested_revision": model_cfg["revision"],
    "resolved_revision": resolved_revision,
    "precision": "float16",
    "run_scope": RUN_SCOPE,
    "case_count": len(cases),
    "prompt_sha256": PROMPT_HASH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "decoding": "greedy",
    "environment": versions,
    "load_metrics": load_metrics,
    "successful_cases": sum(r.get("status") == "SUCCESS" for r in saved_rows),
    "error_cases": sum(r.get("status") != "SUCCESS" for r in saved_rows),
    "mean_generation_seconds": statistics.mean([r["generation_seconds"] for r in saved_rows if "generation_seconds" in r]) if saved_rows else None,
    "max_gpu_peak_allocated_gib": max([r.get("gpu_peak_allocated_gib", 0) for r in saved_rows], default=None),
    "note": "Colab FP16 development measurement; not Android latency and not independent test accuracy",
}
with open(os.path.join(run_dir, "run_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
# 셀 10: strict 채점, 분해 채점, 계약 무시 오류 진단
import subprocess, sys

if RUN_SCOPE != "gate":
    raise ValueError("v3.1 재평가는 먼저 gate 범위로 실행하세요.")
gate_gold = os.path.join(run_dir, "gate_gold.jsonl")
with open(gate_gold, "w", encoding="utf-8") as f:
    for case in cases:
        f.write(json.dumps(case, ensure_ascii=False) + "\n")

score_dir = os.path.join(run_dir, "scores")
subprocess.run([
    sys.executable, os.path.join(EVAL_ROOT, "scripts", "score_predictions.py"),
    "--gold", gate_gold, "--predictions", pred_path, "--out-dir", score_dir,
], check=True)
subprocess.run([
    sys.executable, os.path.join(EVAL_ROOT, "scripts", "score_predictions_v2.py"),
    "--gold", gate_gold, "--predictions", pred_path, "--out-dir", score_dir,
], check=True)
subprocess.run([
    sys.executable, os.path.join(EVAL_ROOT, "scripts", "audit_gate_errors.py"),
    "--gold", gate_gold, "--predictions", pred_path,
    "--out-dir", os.path.join(run_dir, "diagnostics"), "--model-id", model_cfg["model_id"],
], check=True)
print("채점과 오류 진단 완료:", run_dir)


In [ ]:
# 셀 11: 모델 메모리 해제
del model
if processor is not None:
    del processor
if tokenizer is not None:
    del tokenizer
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print("모델 해제 완료, GPU allocated GiB:", torch.cuda.memory_allocated() / 1024**3)


In [ ]:
# 셀 12: 결과 요약과 ZIP 다운로드
import shutil
from google.colab import files

with open(os.path.join(run_dir, "scores", "summary.json"), encoding="utf-8") as f:
    strict_summary = json.load(f)
with open(os.path.join(run_dir, "diagnostics", "error_summary.json"), encoding="utf-8") as f:
    diagnostic_summary = json.load(f)
print("strict summary")
print(json.dumps(strict_summary, ensure_ascii=False, indent=2))
print("diagnostic summary")
print(json.dumps(diagnostic_summary, ensure_ascii=False, indent=2))

zip_path = shutil.make_archive(f"/content/{run_name}", "zip", run_dir)
print("ZIP:", zip_path)
files.download(zip_path)
